In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from sklearn.metrics.pairwise import cosine_similarity
import difflib

##Load Dataset

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
df = pd.read_csv("/content/drive/MyDrive/dataset/movies.csv")

In [7]:
df.head()

,Unnamed: 0,Title,Release Date,Description,Rating,No of Persons Voted,Directed by,Written by,Duration,Genres
0,0,Dekalog (1988),"Mar 22, 1996",This masterwork by Krzysztof Kieślowski is one...,7.4,118,Krzysztof Kieslowski,"Krzysztof Kieslowski, Krzysztof Piesiewicz",9 h 32 m,Drama
1,1,Three Colors: Red,"Nov 23, 1994",Krzysztof Kieslowski closes his Three Colors t...,8.3,241,Krzysztof Kieslowski,"Krzysztof Kieslowski, Krzysztof Piesiewicz, Ag...",1 h 39 m,"Drama,Mystery,Romance"
2,2,The Conformist,"Oct 22, 1970","Set in Rome in the 1930s, this re-release of B...",7.3,106,Bernardo Bertolucci,"Alberto Moravia, Bernardo Bertolucci",1 h 47 m,Drama
3,3,Tokyo Story,"Mar 13, 1972",Yasujiro Ozu’s Tokyo Story follows an aging co...,8.1,147,Yasujirô Ozu,"Kôgo Noda, Yasujirô Ozu",2 h 16 m,Drama
4,4,The Leopard (re-release),"Aug 13, 2004","Set in Sicily in 1860, Luchino Visconti's spec...",7.8,85,Luchino Visconti,"Giuseppe Tomasi di Lampedusa, Suso Cecchi D'Am...",3 h 7 m,"Drama,History"


In [8]:
print(df['Genres'].unique)

<bound method Series.unique of 0                                    Drama
1                    Drama,Mystery,Romance
2                                    Drama
3                                    Drama
4                            Drama,History
                       ...                
16285                                Drama
16286                          Documentary
16287                          Documentary
16288    Documentary,Biography,History,War
16289                          Documentary
Name: Genres, Length: 16290, dtype: object>


In [9]:
df = df.drop(columns=['Unnamed: 0'])

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16290 entries, 0 to 16289
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Title                16290 non-null  object 
 1   Release Date         16290 non-null  object 
 2   Description          16290 non-null  object 
 3   Rating               12846 non-null  float64
 4   No of Persons Voted  12829 non-null  object 
 5   Directed by          16283 non-null  object 
 6   Written by           15327 non-null  object 
 7   Duration             16277 non-null  object 
 8   Genres               16285 non-null  object 
dtypes: float64(1), object(8)
memory usage: 1.1+ MB


In [11]:
df.isnull().sum()

,0
Title,0
Release Date,0
Description,0
Rating,3444
No of Persons Voted,3461
Directed by,7
Written by,963
Duration,13
Genres,5


In [12]:
df.shape

(16290, 9)

In [13]:
# Remove duplicate movie titles
df = df.drop_duplicates(subset=['Title'], keep='first')
print(f"after remove deplication shape: {df.shape}")

after remove deplication shape: (14693, 9)


*Text columns*


In [14]:
df['Written by'] = df['Written by'].fillna(' ')
df['Directed by'] = df['Directed by'].fillna(' ')
df = df.dropna(subset = ['Duration'])
df = df.dropna(subset = ['Genres'])



In [15]:
df.shape

(14676, 9)

*Numeric columns*

In [16]:
df['Rating'] = df['Rating'].fillna(df['Rating'].mean())


In [17]:
df['No of Persons Voted'] = pd.to_numeric(df['No of Persons Voted'], errors='coerce')
df['No of Persons Voted'] = df['No of Persons Voted'].fillna(df['No of Persons Voted'].median())

In [18]:
df.shape

(14676, 9)

In [19]:
df.isnull().sum()

,0
Title,0
Release Date,0
Description,0
Rating,0
No of Persons Voted,0
Directed by,0
Written by,0
Duration,0
Genres,0


##Data cleaning

In [20]:
import re

def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z ]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    return ''

df['Description'] = df['Description'].apply(clean_text)
df['Genres'] = df['Genres'].apply(clean_text)
df['Directed by'] = df['Directed by'].apply(clean_text)
df['Written by'] = df['Written by'].apply(clean_text)

In [21]:
df.head()

,Title,Release Date,Description,Rating,No of Persons Voted,Directed by,Written by,Duration,Genres
0,Dekalog (1988),"Mar 22, 1996",this masterwork by krzysztof kie lowski is one...,7.4,118.0,krzysztof kieslowski,krzysztof kieslowski krzysztof piesiewicz,9 h 32 m,drama
1,Three Colors: Red,"Nov 23, 1994",krzysztof kieslowski closes his three colors t...,8.3,241.0,krzysztof kieslowski,krzysztof kieslowski krzysztof piesiewicz agni...,1 h 39 m,drama mystery romance
2,The Conformist,"Oct 22, 1970",set in rome in the s this re release of bernar...,7.3,106.0,bernardo bertolucci,alberto moravia bernardo bertolucci,1 h 47 m,drama
3,Tokyo Story,"Mar 13, 1972",yasujiro ozu s tokyo story follows an aging co...,8.1,147.0,yasujir ozu,k go noda yasujir ozu,2 h 16 m,drama
4,The Leopard (re-release),"Aug 13, 2004",set in sicily in luchino visconti s spectacula...,7.8,85.0,luchino visconti,giuseppe tomasi di lampedusa suso cecchi d ami...,3 h 7 m,drama history


In [22]:
df['popularity'] = df['Rating'] * df['No of Persons Voted']

**Convert Duration to Minutes**

In [23]:
import re

def convert_to_minutes(duration):
    duration = str(duration).lower()

    hours = 0
    minutes = 0

    h = re.search(r'(\d+)\s*h', duration)
    if h:
        hours = int(h.group(1))

    m = re.search(r'(\d+)\s*m', duration)
    if m:
        minutes = int(m.group(1))

    total_minutes = hours * 60 + minutes
    return total_minutes


df['Duration_minutes'] = df['Duration'].apply(convert_to_minutes)

In [24]:
df.columns

Index(['Title', 'Release Date', 'Description', 'Rating', 'No of Persons Voted',
       'Directed by', 'Written by', 'Duration', 'Genres', 'popularity',
       'Duration_minutes'],
      dtype='object')

In [25]:
df.isnull().sum()

,0
Title,0
Release Date,0
Description,0
Rating,0
No of Persons Voted,0
Directed by,0
Written by,0
Duration,0
Genres,0
popularity,0


In [26]:
df.head()

,Title,Release Date,Description,Rating,No of Persons Voted,Directed by,Written by,Duration,Genres,popularity,Duration_minutes
0,Dekalog (1988),"Mar 22, 1996",this masterwork by krzysztof kie lowski is one...,7.4,118.0,krzysztof kieslowski,krzysztof kieslowski krzysztof piesiewicz,9 h 32 m,drama,873.2,572
1,Three Colors: Red,"Nov 23, 1994",krzysztof kieslowski closes his three colors t...,8.3,241.0,krzysztof kieslowski,krzysztof kieslowski krzysztof piesiewicz agni...,1 h 39 m,drama mystery romance,2000.3,99
2,The Conformist,"Oct 22, 1970",set in rome in the s this re release of bernar...,7.3,106.0,bernardo bertolucci,alberto moravia bernardo bertolucci,1 h 47 m,drama,773.8,107
3,Tokyo Story,"Mar 13, 1972",yasujiro ozu s tokyo story follows an aging co...,8.1,147.0,yasujir ozu,k go noda yasujir ozu,2 h 16 m,drama,1190.7,136
4,The Leopard (re-release),"Aug 13, 2004",set in sicily in luchino visconti s spectacula...,7.8,85.0,luchino visconti,giuseppe tomasi di lampedusa suso cecchi d ami...,3 h 7 m,drama history,663.0,187


COMBINE FEATURES

In [27]:
df['combined_features'] = df['Description'] + ' ' + df['Genres'] + ' ' + df['Directed by'] + ' '+ df['Written by']

In [28]:
df.head()

,Title,Release Date,Description,Rating,No of Persons Voted,Directed by,Written by,Duration,Genres,popularity,Duration_minutes,combined_features
0,Dekalog (1988),"Mar 22, 1996",this masterwork by krzysztof kie lowski is one...,7.4,118.0,krzysztof kieslowski,krzysztof kieslowski krzysztof piesiewicz,9 h 32 m,drama,873.2,572,this masterwork by krzysztof kie lowski is one...
1,Three Colors: Red,"Nov 23, 1994",krzysztof kieslowski closes his three colors t...,8.3,241.0,krzysztof kieslowski,krzysztof kieslowski krzysztof piesiewicz agni...,1 h 39 m,drama mystery romance,2000.3,99,krzysztof kieslowski closes his three colors t...
2,The Conformist,"Oct 22, 1970",set in rome in the s this re release of bernar...,7.3,106.0,bernardo bertolucci,alberto moravia bernardo bertolucci,1 h 47 m,drama,773.8,107,set in rome in the s this re release of bernar...
3,Tokyo Story,"Mar 13, 1972",yasujiro ozu s tokyo story follows an aging co...,8.1,147.0,yasujir ozu,k go noda yasujir ozu,2 h 16 m,drama,1190.7,136,yasujiro ozu s tokyo story follows an aging co...
4,The Leopard (re-release),"Aug 13, 2004",set in sicily in luchino visconti s spectacula...,7.8,85.0,luchino visconti,giuseppe tomasi di lampedusa suso cecchi d ami...,3 h 7 m,drama history,663.0,187,set in sicily in luchino visconti s spectacula...


In [29]:
df = df.reset_index(drop=True)

## TF-IDF VECTORIZATION

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.8,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(df['combined_features'])

In [31]:
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 477181 stored elements and shape (14676, 5000)>
  Coords	Values
  (0, 4618)	0.15311030937979167
  (0, 653)	0.09744304805560211
  (0, 1952)	0.11738346683216727
  (0, 4763)	0.15156962180705325
  (0, 4261)	0.16138116810184103
  (0, 3423)	0.1621647963112342
  (0, 4405)	0.1322271637468318
  (0, 1736)	0.13789900340231162
  (0, 3702)	0.13634038639266538
  (0, 2176)	0.1621647963112342
  (0, 873)	0.2145352634895137
  (0, 2573)	0.11231469894556556
  (0, 859)	0.1621647963112342
  (0, 2691)	0.13321596465465216
  (0, 2300)	0.15655266554819608
  (0, 1552)	0.10169102269856324
  (0, 1404)	0.12024300881275532
  (0, 1113)	0.12528715002694654
  (0, 3341)	0.0996595216879409
  (0, 2183)	0.10278056134201469
  (0, 2170)	0.14552068274120447
  (0, 2709)	0.0921457223292368
  (0, 1694)	0.16006477608001374
  (0, 1311)	0.15594081597406445
  (0, 2277)	0.14590242045514584
  :	:
  (14674, 2111)	0.14793997226161376
  (14674, 429)	0.14524039032895292
  (1467

COSINE SIMILARITY

In [32]:
similarity = cosine_similarity(tfidf_matrix)


In [33]:
similarity.shape

(14676, 14676)

In [34]:
print(similarity)

[[1.         0.00520973 0.00231537 ... 0.04314416 0.         0.03284547]
 [0.00520973 1.         0.07233492 ... 0.         0.         0.        ]
 [0.00231537 0.07233492 1.         ... 0.         0.         0.        ]
 ...
 [0.04314416 0.         0.         ... 1.         0.02466904 0.02083914]
 [0.         0.         0.         ... 0.02466904 1.         0.05948827]
 [0.03284547 0.         0.         ... 0.02083914 0.05948827 1.        ]]


## RECOMMENDATION FUNCTION

In [35]:
'''def recommend(movie_title, num_recommendations=10):
    movie_title = movie_title.lower()
    matches = df[df['Title'].str.lower().str.contains(movie_title)]

    if matches.empty:
        print("\n❌ Movie not found. Try another name.")
        return

    pos_index = matches.index[0]   # now a clean positional index (0..n-1)
    print(f"\n🎬 Recommendations based on: {df.iloc[pos_index]['Title']}\n")

    # Use full similarity if already computed, else compute on the fly
    if 'similarity' in globals():
        sim_scores = similarity[pos_index]
    else:
        query_vec = tfidf_matrix[pos_index]
        sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Get top N (skip the first because it's the movie itself)
    top_indices = sim_scores.argsort()[::-1][1:num_recommendations+1]

    for i, idx in enumerate(top_indices, 1):
        print(f"{i}. {df.iloc[idx]['Title']} (Score: {sim_scores[idx]:.3f})")'''

'def recommend(movie_title, num_recommendations=10):\n    movie_title = movie_title.lower()\n    matches = df[df[\'Title\'].str.lower().str.contains(movie_title)]\n\n    if matches.empty:\n        print("\n❌ Movie not found. Try another name.")\n        return\n\n    pos_index = matches.index[0]   # now a clean positional index (0..n-1)\n    print(f"\n🎬 Recommendations based on: {df.iloc[pos_index][\'Title\']}\n")\n\n    # Use full similarity if already computed, else compute on the fly\n    if \'similarity\' in globals():\n        sim_scores = similarity[pos_index]\n    else:\n        query_vec = tfidf_matrix[pos_index]\n        sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()\n\n    # Get top N (skip the first because it\'s the movie itself)\n    top_indices = sim_scores.argsort()[::-1][1:num_recommendations+1]\n\n    for i, idx in enumerate(top_indices, 1):\n        print(f"{i}. {df.iloc[idx][\'Title\']} (Score: {sim_scores[idx]:.3f})")'

In [36]:
'''recommend("superman")
  # if it exists'''

'recommend("superman")\n  # if it exists'

enter genre and find movie

In [37]:
'''def recommend(genre_title, num_recommendations=10):
    genre_title = genre_title.lower()
    matches = df[df['Genres'].str.lower().str.contains(genre_title)]

    if matches.empty:
        print("\n❌ Movie not found. Try another name.")
        return

    pos_index = matches.index[0]   # now a clean positional index (0..n-1)
    print(f"\n🎬 Recommendations based on: {df.iloc[pos_index]['Genres']}\n")

    # Use full similarity if already computed, else compute on the fly
    if 'similarity' in globals():
        sim_scores = similarity[pos_index]
    else:
        query_vec = tfidf_matrix[pos_index]
        sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Get top N (skip the first because it's the movie itself)
    top_indices = sim_scores.argsort()[::-1][1:num_recommendations+1]

    for i, idx in enumerate(top_indices, 1):
        print(f"{i}. {df.iloc[idx]['Title']} (Score: {sim_scores[idx]:.3f})")'''

'def recommend(genre_title, num_recommendations=10):\n    genre_title = genre_title.lower()\n    matches = df[df[\'Genres\'].str.lower().str.contains(genre_title)]\n\n    if matches.empty:\n        print("\n❌ Movie not found. Try another name.")\n        return\n\n    pos_index = matches.index[0]   # now a clean positional index (0..n-1)\n    print(f"\n🎬 Recommendations based on: {df.iloc[pos_index][\'Genres\']}\n")\n\n    # Use full similarity if already computed, else compute on the fly\n    if \'similarity\' in globals():\n        sim_scores = similarity[pos_index]\n    else:\n        query_vec = tfidf_matrix[pos_index]\n        sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()\n\n    # Get top N (skip the first because it\'s the movie itself)\n    top_indices = sim_scores.argsort()[::-1][1:num_recommendations+1]\n\n    for i, idx in enumerate(top_indices, 1):\n        print(f"{i}. {df.iloc[idx][\'Title\']} (Score: {sim_scores[idx]:.3f})")'

In [38]:
'''recommend("adventure")
'''

'recommend("adventure")\n'

In [39]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend1(movie_title, num_recommendations=10):
    movie_title = movie_title.lower()
    matches = df[df['Title'].str.lower().str.contains(movie_title)]

    if matches.empty:
        print("\n❌ Movie not found. Try another name.")
        return

    pos_index = matches.index[0]

    print(f"\n🎬 Recommendations based on : (Genre: {df.iloc[pos_index]['Genres']})\n")

    # Get similarity scores
    if 'similarity' in globals():
        sim_scores = similarity[pos_index]
    else:
        query_vec = tfidf_matrix[pos_index]
        sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # ✅ INCLUDE the searched movie (no skipping)
    top_indices = sim_scores.argsort()[::-1][:num_recommendations]

    for i, idx in enumerate(top_indices, 1):
        print(f"{i}. {df.iloc[idx]['Title']} (Score: {sim_scores[idx]:.3f})")

In [40]:
recommend1(input("Enter Movie Name :"))



🎬 Recommendations based on : (Genre: drama)

1. Dekalog (1988) (Score: 1.000)
2. House of Pleasures (Score: 0.153)
3. Norwegian Wood (Score: 0.149)
4. Daybreak (Score: 0.148)
5. The Nines (Score: 0.144)
6. The Double Life of Veronique (Score: 0.138)
7. The Wind and the Lion (Score: 0.135)
8. Freud's Last Session (Score: 0.130)
9. Avalon (Score: 0.130)
10. Tuesday, After Christmas (Score: 0.129)


In [41]:
import pickle

model_data = {
    "movies": df,
    "similarity": similarity
}

with open("movie_recommender.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("✅ PKL file created successfully!")

✅ PKL file created successfully!


In [42]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'drive', 'movie_recommender.pkl', 'sample_data']


In [43]:
#save pkl inside colab

with open("/content/drive/MyDrive/dataset/movie_recommender.pkl", "wb") as f:
    pickle.dump(model_data, f)